# 43 — Ensemble (voting + stacking)

Combina os 4 modelos base do artigo (decisao H):

| Base model | Notebook |
|---|---|
| `tfidf_logreg` | 11 |
| `tfidf_linearsvc` | 12 |
| `tfidf_multinomialnb` | 13 |
| `bert_bertimbau_base` | 21 |

**Estrategias** (binario):
- `majority_vote` — pelo menos 3 dos 4 votam 1
- `weighted_vote` — score medio ponderado por F1, threshold 0.5
- `stacking` — meta-classificador (LogReg) treinado nos scores da val, aplicado em test (so para `test_set`)

**Estrategias** (multiclasse):
- `majority` — classe mais votada (mode); empate vai para a classe predita pelo modelo individual com maior macro_F1
- `stacking` — meta-classificador (LogReg multinomial) treinado sobre as probabilidades por classe (`y_proba_<classe>`) de cada base model na val, aplicado em test (so para `test_set`)

Cada estrategia emite um `result_card.json` com `model_id` no formato `ensemble_{strategy}` e custo = soma dos custos dos base models. Stacking tambem persiste `meta_classifier.joblib` + `meta_classifier_meta.json` na run dir.

**Pre-requisito**: notebooks 11, 12, 13, 21 executados com a versao corrente — `predictions.csv` multiclasse precisa conter colunas `y_proba_<classe>` (caso contrario o stacking multiclasse e pulado).

## 0. Bootstrap (Colab + local)

Detecta o ambiente, monta o Drive em Colab, clona o repo e instala o pacote, e seleciona `RUNS_BASE` apropriado.

- **Colab**: le os `result_card.json` e `predictions.csv` direto de `My Drive/economy-classifier/runs/` (onde 21_bert e os notebooks TF-IDF escrevem). Ensembles novos sao gravados em `My Drive/economy-classifier/runs/ensemble_*/`.
- **Local**: usa `artifacts/runs/`.

In [1]:
import subprocess
import sys
import zipfile
from pathlib import Path


def _run(cmd: list[str], description: str) -> None:
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr, file=sys.stderr)
        raise RuntimeError(f"{description} failed with exit code {result.returncode}")


IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/almeidadm/economy-classifier.git"
    REPO_BRANCH = "main"
    DRIVE_FOLDER = "economy-classifier"

    DRIVE_BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/economy-classifier")

    if REPO_DIR.exists():
        _run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], "git fetch")
        _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], "git checkout")
        _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], "git reset")
    else:
        _run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], "git clone")

    _run(
        [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR),
         "--upgrade-strategy", "only-if-needed", "-q"],
        "pip install -e .",
    )
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        zip_path = DRIVE_BASE / "colab_splits.zip"
        assert zip_path.exists(), (
            f"Falta colab_splits.zip em {DRIVE_BASE}. Rode scripts/colab_pack.py local e suba o zip."
        )
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO_DIR / "artifacts")
        print("Splits extraidos em", SPLITS_DIR)

    RUNS_BASE = DRIVE_BASE / "runs"
    HARDWARE = "Colab-CPU"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"
    HARDWARE = "Local-CPU"

assert RUNS_BASE.exists(), (
    f"RUNS_BASE nao encontrado: {RUNS_BASE}. "
    "No Colab, verifique que 11-21 ja gravaram em My Drive/economy-classifier/runs/. "
    "Local: rode os notebooks 11-21 ou desempacote com scripts/colab_unpack_streaming.py."
)
RUNS_BASE.mkdir(parents=True, exist_ok=True)
print("SPLITS_DIR:", SPLITS_DIR)
print("RUNS_BASE :", RUNS_BASE)
print("HARDWARE  :", HARDWARE)

Ambiente: Local
SPLITS_DIR: /home/diacrono/Documentos/repositorios/economy-classifier/artifacts/splits
RUNS_BASE : /home/diacrono/Documentos/repositorios/economy-classifier/artifacts/runs
HARDWARE  : Local-CPU


## 1. Imports e configuracao

In [2]:
import json, time
from collections import Counter

import numpy as np
import pandas as pd

from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL
from economy_classifier.ensemble import (
    majority_vote, weighted_vote, train_stacking_classifier, predict_stacking,
    save_stacking_classifier,
    compute_agreement_matrix, compute_fleiss_kappa,
)
from economy_classifier.evaluation import (
    compute_binary_metrics, compute_confusion_matrix,
    compute_cost_metrics, compute_multiclass_metrics, compute_roc_auc,
    summarize_cv_metrics,
)
from economy_classifier.project import build_result_card, persist_result_card

BASE_MODELS = [
    "tfidf_logreg",
    "tfidf_linearsvc",
    "tfidf_multinomialnb",
    "bert_bertimbau_base",
]
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

## 2. Loaders e helpers de alinhamento

In [3]:
def card_path(model_id, task, regime):
    return RUNS_BASE / f"{model_id}_{task}_{regime}" / "result_card.json"

def predictions_path(model_id, task, regime):
    return RUNS_BASE / f"{model_id}_{task}_{regime}" / "predictions.csv"

def load_card(model_id, task, regime):
    p = card_path(model_id, task, regime)
    return json.loads(p.read_text()) if p.exists() else None

def load_preds(model_id, task, regime):
    p = predictions_path(model_id, task, regime)
    return pd.read_csv(p) if p.exists() else None

def align_by_index(preds_by_model, fold=None):
    base = None
    for model_id, df in preds_by_model.items():
        if fold is not None and "fold" in df.columns:
            df = df[df["fold"] == fold]
        df = df.set_index("index")
        cols = {"y_pred": f"y_pred_{model_id}"}
        if "y_score" in df.columns:
            cols["y_score"] = f"y_score_{model_id}"
        df = df.rename(columns=cols)
        keep = ["y_true", f"y_pred_{model_id}"]
        if f"y_score_{model_id}" in df.columns:
            keep.append(f"y_score_{model_id}")
        df = df[keep]
        base = df if base is None else base.join(df.drop(columns=["y_true"]), how="inner")
    return base.reset_index()

def align_multiclass_proba(preds_by_model, expected_labels):
    """Align multiclass predictions across base models on the shared index.

    Returns ``(meta_df, proba_per_model)`` where:
    - ``meta_df`` carries ``index``, ``y_true`` (taken from the first model);
    - ``proba_per_model`` is ``{model_id: DataFrame[expected_labels]}`` with
      one row per aligned sample. Missing ``y_proba_<class>`` columns
      raise so the caller can skip cleanly.
    """
    proba_per_model = {}
    base = None
    for model_id, df in preds_by_model.items():
        missing = [c for c in expected_labels if f"y_proba_{c}" not in df.columns]
        if missing:
            raise KeyError(f"{model_id} missing y_proba columns: {missing}")
        local = df.set_index("index")
        # Keep y_true once (first model); drop later to avoid duplicates on join.
        slim = local[[f"y_proba_{c}" for c in expected_labels]].copy()
        slim.columns = expected_labels  # strip prefix; model_id will scope it
        if base is None:
            base = local[["y_true"]].copy()
        proba_per_model[model_id] = slim
        base = base.join(slim.add_prefix(f"_{model_id}__"), how="inner")
    aligned_index = base.index
    proba_per_model = {m: df.loc[aligned_index].reset_index(drop=True) for m, df in proba_per_model.items()}
    meta_df = base.loc[aligned_index, ["y_true"]].reset_index()
    return meta_df, proba_per_model

def cost_sum_from_cards(cards):
    return {
        "train_total": sum((c["cost"].get("train_seconds_mean") or 0.0) for c in cards.values()),
        "inf_total":   sum((c["cost"].get("inference_seconds_mean") or 0.0) for c in cards.values()),
        "size_total":  sum((c["cost"].get("model_size_mb") or 0.0) for c in cards.values()),
    }

## 3. BINARIO — voting (3 regimes)

Pesos do `weighted_vote` = F1 do regime correspondente do modelo base. Threshold fixo em 0.5.

In [4]:
def f1_from_card(card):
    m = card["metrics"]
    return m.get("f1") or m.get("f1_mean") or 0.0

def run_binary_voting(regime):
    cards = {m: load_card(m, "binary", regime) for m in BASE_MODELS}
    preds = {m: load_preds(m, "binary", regime) for m in BASE_MODELS}
    if any(c is None or p is None for c, p in zip(cards.values(), preds.values())):
        missing = [m for m in BASE_MODELS if cards[m] is None or preds[m] is None]
        print(f"  SKIP {regime}: cards faltando para {missing}")
        return
    weights = {m: f1_from_card(cards[m]) for m in BASE_MODELS}
    cost_base = cost_sum_from_cards(cards)
    folds = sorted(preds[BASE_MODELS[0]]["fold"].unique()) if regime == "cv_5fold" else [None]

    for strategy in ["majority", "weighted"]:
        fold_metrics, all_out = [], []
        for fold in folds:
            aligned = align_by_index(preds, fold=fold)
            if strategy == "majority":
                pred_dict = {m: aligned[f"y_pred_{m}"] for m in BASE_MODELS}
                out = majority_vote(pred_dict, threshold=3)
                y_score = pd.Series(np.nan, index=aligned.index)
            else:
                score_dict = {m: aligned[f"y_score_{m}"] for m in BASE_MODELS if f"y_score_{m}" in aligned.columns}
                out = weighted_vote(score_dict, weights, threshold=0.5)
                y_score = out["y_score"]
            metrics = compute_binary_metrics(aligned["y_true"].values, out["y_pred"].values)
            if y_score.notna().all():
                metrics["auc_roc"] = round(compute_roc_auc(aligned["y_true"].values, y_score.values), 4)
            fold_metrics.append(metrics)
            df = pd.DataFrame({
                "index": aligned["index"], "y_true": aligned["y_true"],
                "y_pred": out["y_pred"], "method": f"ensemble_{strategy}",
            })
            if y_score.notna().all(): df["y_score"] = y_score
            if fold is not None: df["fold"] = fold
            all_out.append(df)
        agg = summarize_cv_metrics(fold_metrics) if regime == "cv_5fold" else fold_metrics[0]
        cost = compute_cost_metrics(
            train_seconds=cost_base["train_total"], inference_seconds=cost_base["inf_total"],
            n_inference_samples=len(all_out[0]),
            model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
        )
        run_dir = RUNS_BASE / f"ensemble_{strategy}_binary_{regime}"
        run_dir.mkdir(parents=True, exist_ok=True)
        pd.concat(all_out).to_csv(run_dir / "predictions.csv", index=False)
        card = build_result_card(
            model_id=f"ensemble_{strategy}", task="binary", regime=regime,
            metrics=agg, cost=cost,
            config={"base_models": BASE_MODELS,
                    "weights": weights if strategy == "weighted" else None,
                    "threshold": 0.5 if strategy == "weighted" else 3},
            n_train_samples=None, n_eval_samples=len(all_out[0]),
            predictions_path=str(run_dir / "predictions.csv"),
            notes=f"{strategy} vote across {len(BASE_MODELS)} base models",
        )
        persist_result_card(card, run_dir)
        primary = agg.get("f1") or agg.get("f1_mean")
        print(f"  {strategy:9s} {regime:14s} f1={primary}")

for regime in ["fixed_split", "cv_5fold", "test_set"]:
    print(f"\n=== BINARY {regime} ===")
    run_binary_voting(regime)


=== BINARY fixed_split ===
  SKIP fixed_split: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']

=== BINARY cv_5fold ===
  SKIP cv_5fold: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']

=== BINARY test_set ===
  SKIP test_set: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']


## 4. BINARIO — stacking (apenas `test_set`)

Meta treinado nos scores da `fixed_split` val, aplicado nas predicoes de `test_set` dos base models.

In [5]:
cards_fs = {m: load_card(m, "binary", "fixed_split") for m in BASE_MODELS}
cards_ts = {m: load_card(m, "binary", "test_set") for m in BASE_MODELS}
preds_val = {m: load_preds(m, "binary", "fixed_split") for m in BASE_MODELS}
preds_test = {m: load_preds(m, "binary", "test_set") for m in BASE_MODELS}

if any(v is None for v in {**cards_fs, **cards_ts, **preds_val, **preds_test}.values()):
    print("SKIP stacking: result cards de fixed_split ou test_set faltando.")
else:
    aligned_val = align_by_index(preds_val)
    aligned_test = align_by_index(preds_test)
    val_scores = {m: aligned_val[f"y_score_{m}"] for m in BASE_MODELS}
    test_scores = {m: aligned_test[f"y_score_{m}"] for m in BASE_MODELS}
    t0 = time.perf_counter()
    meta = train_stacking_classifier(val_scores, aligned_val["y_true"], seed=2026)
    meta_train_s = time.perf_counter() - t0
    out = predict_stacking(meta, test_scores)
    metrics = compute_binary_metrics(aligned_test["y_true"].values, out["y_pred"].values)
    metrics["auc_roc"] = round(compute_roc_auc(aligned_test["y_true"].values, out["y_score"].values), 4)
    cost_base = cost_sum_from_cards(cards_ts)
    cost = compute_cost_metrics(
        train_seconds=cost_base["train_total"] + meta_train_s,
        inference_seconds=cost_base["inf_total"],
        n_inference_samples=len(aligned_test),
        model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
    )
    run_dir = RUNS_BASE / "ensemble_stacking_binary_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "index": aligned_test["index"], "y_true": aligned_test["y_true"],
        "y_pred": out["y_pred"], "y_score": out["y_score"],
        "method": "ensemble_stacking",
    }).to_csv(run_dir / "predictions.csv", index=False)
    save_stacking_classifier(meta, run_dir, feature_names=BASE_MODELS)
    card = build_result_card(
        model_id="ensemble_stacking", task="binary", regime="test_set",
        metrics=metrics, cost=cost,
        config={"base_models": BASE_MODELS, "meta": "LogisticRegression",
                "meta_trained_on": "binary_fixed_split val scores"},
        n_train_samples=len(aligned_val), n_eval_samples=len(aligned_test),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Stacking meta (LogReg) trained on val of fixed_split; applied to test_set scores",
    )
    persist_result_card(card, run_dir)
    print("BINARY stacking test_set:", json.dumps(metrics, indent=2))

SKIP stacking: result cards de fixed_split ou test_set faltando.


## 5. MULTICLASSE — majority vote (3 regimes)

Empate vai para a classe predita pelo modelo individual com maior macro_F1.

In [6]:
def macro_f1_from_card(card):
    m = card["metrics"]
    return m.get("macro_f1") or m.get("macro_f1_mean") or 0.0

def multiclass_majority(preds_wide, model_order_by_f1):
    cols = [f"y_pred_{m}" for m in model_order_by_f1]
    out = []
    for _, row in preds_wide[cols].iterrows():
        counts = Counter(row.tolist())
        max_v = max(counts.values())
        candidates = [c for c, v in counts.items() if v == max_v]
        if len(candidates) == 1:
            out.append(candidates[0])
        else:
            for c in cols:
                if row[c] in candidates:
                    out.append(row[c]); break
    return pd.Series(out, index=preds_wide.index)

def run_multi_voting(regime):
    cards = {m: load_card(m, "multiclass", regime) for m in BASE_MODELS}
    preds = {m: load_preds(m, "multiclass", regime) for m in BASE_MODELS}
    if any(c is None or p is None for c, p in zip(cards.values(), preds.values())):
        missing = [m for m in BASE_MODELS if cards[m] is None or preds[m] is None]
        print(f"  SKIP {regime}: cards faltando para {missing}")
        return
    order = sorted(BASE_MODELS, key=lambda m: macro_f1_from_card(cards[m]), reverse=True)
    cost_base = cost_sum_from_cards(cards)
    folds = sorted(preds[BASE_MODELS[0]]["fold"].unique()) if regime == "cv_5fold" else [None]
    fold_metrics, all_out = [], []
    for fold in folds:
        aligned = align_by_index(preds, fold=fold)
        y_pred = multiclass_majority(aligned, order)
        m = compute_multiclass_metrics(aligned["y_true"], y_pred, labels=MULTI_LABELS)
        fold_metrics.append(m)
        df = pd.DataFrame({
            "index": aligned["index"], "y_true": aligned["y_true"],
            "y_pred": y_pred, "method": "ensemble_majority_multi",
        })
        if fold is not None: df["fold"] = fold
        all_out.append(df)
    agg = summarize_cv_metrics(fold_metrics) if regime == "cv_5fold" else fold_metrics[0]
    cost = compute_cost_metrics(
        train_seconds=cost_base["train_total"], inference_seconds=cost_base["inf_total"],
        n_inference_samples=len(all_out[0]),
        model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
    )
    run_dir = RUNS_BASE / f"ensemble_majority_multiclass_{regime}"
    run_dir.mkdir(parents=True, exist_ok=True)
    pd.concat(all_out).to_csv(run_dir / "predictions.csv", index=False)
    if regime != "cv_5fold":
        cm = compute_confusion_matrix(all_out[0]["y_true"], all_out[0]["y_pred"], labels=MULTI_LABELS)
        cm.to_csv(run_dir / "confusion_matrix.csv")
    card = build_result_card(
        model_id="ensemble_majority", task="multiclass", regime=regime,
        metrics=agg, cost=cost,
        config={"base_models": BASE_MODELS, "tiebreak_order": order},
        n_train_samples=None, n_eval_samples=len(all_out[0]),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Mode across base models; tiebreak by descending individual macro_F1",
    )
    persist_result_card(card, run_dir)
    primary = agg.get("macro_f1") or agg.get("macro_f1_mean")
    print(f"  majority {regime:14s} macro_f1={primary}")

for regime in ["fixed_split", "cv_5fold", "test_set"]:
    print(f"\n=== MULTI {regime} ===")
    run_multi_voting(regime)


=== MULTI fixed_split ===
  SKIP fixed_split: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']

=== MULTI cv_5fold ===
  SKIP cv_5fold: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']

=== MULTI test_set ===
  SKIP test_set: cards faltando para ['tfidf_logreg', 'tfidf_linearsvc', 'tfidf_multinomialnb', 'bert_bertimbau_base']


## 6. MULTICLASSE — stacking (apenas `test_set`)

Meta-classificador (LogReg multinomial) treinado nas probabilidades por classe (`y_proba_<classe>`) dos 4 base models na val (`fixed_split`), aplicado em test. Features finais: 4 base models x 8 classes = 32 colunas. Pula limpo se algum `predictions.csv` multiclasse nao tiver as colunas `y_proba_*` (notebook 11/12/13/21 precisa ter sido re-executado com a versao que emite probas).

In [ ]:
cards_mfs = {m: load_card(m, "multiclass", "fixed_split") for m in BASE_MODELS}
cards_mts = {m: load_card(m, "multiclass", "test_set") for m in BASE_MODELS}
preds_mval = {m: load_preds(m, "multiclass", "fixed_split") for m in BASE_MODELS}
preds_mtest = {m: load_preds(m, "multiclass", "test_set") for m in BASE_MODELS}

missing = [m for m in BASE_MODELS if cards_mfs[m] is None or cards_mts[m] is None
           or preds_mval[m] is None or preds_mtest[m] is None]
if missing:
    print(f"SKIP multi-stacking: cards/preds faltando para {missing}.")
else:
    try:
        meta_val, proba_val = align_multiclass_proba(preds_mval, MULTI_LABELS)
        meta_test, proba_test = align_multiclass_proba(preds_mtest, MULTI_LABELS)
    except KeyError as e:
        print(f"SKIP multi-stacking: {e}. "
              "Re-execute 11/12/13/21 para emitir y_proba_<classe> nos predictions.csv multiclasse.")
    else:
        t0 = time.perf_counter()
        meta = train_stacking_classifier(proba_val, meta_val["y_true"], seed=2026)
        meta_train_s = time.perf_counter() - t0
        out = predict_stacking(meta, proba_test)
        metrics = compute_multiclass_metrics(meta_test["y_true"], out["y_pred"], labels=MULTI_LABELS)
        cost_base = cost_sum_from_cards(cards_mts)
        cost = compute_cost_metrics(
            train_seconds=cost_base["train_total"] + meta_train_s,
            inference_seconds=cost_base["inf_total"],
            n_inference_samples=len(meta_test),
            model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
        )
        run_dir = RUNS_BASE / "ensemble_stacking_multiclass_test_set"
        run_dir.mkdir(parents=True, exist_ok=True)
        out_df = pd.DataFrame({
            "index": meta_test["index"], "y_true": meta_test["y_true"],
            "y_pred": out["y_pred"], "method": "ensemble_stacking_multi",
        })
        proba_cols = [c for c in out.columns if c.startswith("y_proba_")]
        for c in proba_cols:
            out_df[c] = out[c].values
        out_df.to_csv(run_dir / "predictions.csv", index=False)
        cm = compute_confusion_matrix(meta_test["y_true"], out["y_pred"], labels=MULTI_LABELS)
        cm.to_csv(run_dir / "confusion_matrix.csv")
        save_stacking_classifier(meta, run_dir, feature_names=list(meta.feature_names_in_))
        card = build_result_card(
            model_id="ensemble_stacking", task="multiclass", regime="test_set",
            metrics=metrics, cost=cost,
            config={"base_models": BASE_MODELS, "meta": "LogisticRegression(multinomial)",
                    "meta_trained_on": "multiclass_fixed_split val probas",
                    "n_features": int(meta.n_features_in_),
                    "classes": list(meta.classes_)},
            n_train_samples=len(meta_val), n_eval_samples=len(meta_test),
            predictions_path=str(run_dir / "predictions.csv"),
            notes=("Stacking multinomial sobre y_proba_<classe> dos 4 base models. "
                   "Treinado na val de fixed_split, aplicado em test_set."),
        )
        persist_result_card(card, run_dir)
        print(f"MULTI stacking test_set: macro_f1={metrics.get('macro_f1')} "
              f"weighted_f1={metrics.get('weighted_f1')} accuracy={metrics.get('accuracy')}")

## 7. Concordancia entre base models — persistencia em disco

Calcula Cohen's Kappa pareado e Fleiss' Kappa global para os 4 base models, em cada combinacao `(task, regime)`. Persiste:

- `agreement_matrix.csv` — Cohen pareado (NxN).
- `fleiss_kappa.json` — Fleiss kappa global + metadados (categorias, n_subjects, n_raters).
- `contingency_table.csv` — agregacao agreement_level x y_true (so binario).

Diretorio destino: `RUNS_BASE/ensemble_agreement_<task>_<regime>/`. Os arquivos sao recuperaveis pelo `colab_unpack_streaming.py` (whitelist atualizado) para reconstrucao da analise local.

In [7]:
def persist_agreement(task, regime, expected_models=BASE_MODELS):
    """Compute Cohen + Fleiss + (binary only) contingency, save to RUNS_BASE.

    Returns a small summary dict, or None if any required predictions file
    is missing.
    """
    preds = {m: load_preds(m, task, regime) for m in expected_models}
    if any(p is None for p in preds.values()):
        missing = [m for m, p in preds.items() if p is None]
        print(f"  SKIP {task:10s} {regime:14s}: predictions faltando para {missing}")
        return None

    aligned = align_by_index(preds)  # fold=None: para cv_5fold isso une os 5 folds
    pred_dict = {m: aligned[f"y_pred_{m}"] for m in expected_models}

    cohen = compute_agreement_matrix(pred_dict)
    if task == "binary":
        categories = [0, 1]
    else:
        categories = MULTI_LABELS
    fleiss = compute_fleiss_kappa(pred_dict, categories=categories)

    out_dir = RUNS_BASE / f"ensemble_agreement_{task}_{regime}"
    out_dir.mkdir(parents=True, exist_ok=True)
    cohen.round(4).to_csv(out_dir / "agreement_matrix.csv")
    fleiss_payload = {
        "fleiss_kappa": round(fleiss, 4),
        "task": task,
        "regime": regime,
        "n_subjects": int(len(aligned)),
        "n_raters": int(len(expected_models)),
        "raters": list(expected_models),
        "categories": list(categories),
    }
    (out_dir / "fleiss_kappa.json").write_text(
        json.dumps(fleiss_payload, indent=2, ensure_ascii=False),
    )

    if task == "binary":
        from economy_classifier.ensemble import compute_contingency_table
        ct = compute_contingency_table(pred_dict, aligned["y_true"])
        ct.to_csv(out_dir / "contingency_table.csv")

    print(f"  OK   {task:10s} {regime:14s} fleiss={fleiss:.4f} "
          f"n={len(aligned)} -> {out_dir.relative_to(RUNS_BASE.parent)}")
    return fleiss_payload

agreement_summary = []
for task in ["binary", "multiclass"]:
    for regime in ["fixed_split", "cv_5fold", "test_set"]:
        s = persist_agreement(task, regime)
        if s is not None:
            agreement_summary.append(s)

if agreement_summary:
    pd.DataFrame(agreement_summary)[["task", "regime", "fleiss_kappa", "n_subjects", "n_raters"]]
else:
    print("Nenhum bloco de concordancia foi persistido — predictions dos base models faltando.")

SKIP agreement: predictions faltando.


## 8. Sumario de cards do ensemble

In [8]:
rows = []
for sub in sorted(RUNS_BASE.glob("ensemble_*")):
    cp = sub / "result_card.json"
    if not cp.exists(): continue
    c = json.loads(cp.read_text())
    m = c["metrics"]
    primary = m.get("f1") or m.get("f1_mean") or m.get("macro_f1") or m.get("macro_f1_mean")
    rows.append({
        "strategy": c["model_id"], "task": c["task"], "regime": c["regime"],
        "primary": primary,
        "train_total_s": c["cost"].get("train_seconds_mean"),
        "size_total_mb": c["cost"].get("model_size_mb"),
    })
if not rows:
    print("Nenhum ensemble persistido em RUNS_BASE — rode as celulas 3-5 primeiro (com os result_cards dos base models presentes).")
    summary_df = pd.DataFrame()
else:
    summary_df = pd.DataFrame(rows).sort_values(["task", "regime", "strategy"]).reset_index(drop=True)
summary_df

Nenhum ensemble persistido em RUNS_BASE — rode as celulas 3-5 primeiro (com os result_cards dos base models presentes).


""
